# 01. RAG 개념 이해

## RAG란?

**RAG(Retrieval-Augmented Generation)** 는 LLM의 한계를 극복하기 위해  
**외부 문서를 검색(Retrieval)** 하고, 그 결과를 바탕으로 **답변을 생성(Generation)** 하는 기술임

---

## LLM의 한계

| 문제 | 설명 |
|---|---|
| 지식 단절 | 학습 데이터 이후의 최신 정보를 모름 |
| 할루시네이션 | 사실과 다른 내용을 그럴듯하게 생성 |
| 출처 불명 | 어디서 얻은 정보인지 알 수 없음 |
| 컨텍스트 제한 | 모든 문서를 프롬프트에 넣을 수 없음 |

---

## RAG 파이프라인

```
【 인덱싱 단계 (오프라인) 】

  문서 로딩  →  텍스트 분할  →  임베딩  →  벡터 스토어 저장

【 검색·생성 단계 (온라인) 】

  사용자 질문  →  임베딩  →  유사 문서 검색  →  LLM 답변 생성
```

---



## Naive RAG 구성 요소

| 단계 | 역할 | 도구 |
|---|---|---|
| Document Loader | 문서 불러오기 | TextLoader, PyPDFLoader |
| Text Splitter | 문서를 청크로 분할 | RecursiveCharacterTextSplitter |
| Embedding | 텍스트 → 벡터 변환 | OpenAIEmbeddings |
| Vector Store | 벡터 저장 및 검색 | FAISS, Chroma |
| Retriever | 관련 문서 검색 | as_retriever() |
| LLM | 최종 답변 생성 | ChatOpenAI |

## RAG vs 일반 LLM 비교 실습

In [1]:
# from dotenv import load_dotenv
# load_dotenv(override=True, dotenv_path="../.env")

# from langchain_openai import ChatOpenAI

# llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
import sys
sys.path.append("..") # 상위 폴더 path설정
from common_config import llm_connect
model = "gpt-5.4-mini"
llm = llm_connect(model=model)

In [2]:
# LLM 단독 호출 - 모델이 알고 있는 내용만 답변
question = "LangGraph는 무엇이고 어떤 개념을 사용하나요?"

response = llm.invoke(question)
print("[LLM 단독 답변]")
print(response.content)

[LLM 단독 답변]
LangGraph는 **LLM 애플리케이션을 그래프 형태로 설계하고 실행하는 프레임워크**입니다.  
LangChain이 주로 “체인(chain)” 중심이라면, LangGraph는 **상태(state)를 가지고 분기/반복/병렬 흐름을 관리**할 수 있게 해줍니다.

## 핵심 개념

### 1) 그래프(Graph)
- 애플리케이션의 실행 흐름을 **노드(node)**와 **엣지(edge)**로 표현합니다.
- 단순한 일직선 흐름이 아니라:
  - 조건에 따라 분기
  - 루프(반복)
  - 여러 단계의 에이전트 협업
  - 인간 개입(human-in-the-loop)
  같은 복잡한 흐름을 만들 수 있습니다.

### 2) 노드(Node)
- 그래프 안의 하나의 처리 단계입니다.
- 예:
  - 사용자 입력 분석
  - LLM 호출
  - 검색 도구 실행
  - 결과 검증
- 보통 노드는 **입력 상태를 받아서 일부 상태를 업데이트**합니다.

### 3) 엣지(Edge)
- 노드와 노드를 연결하는 경로입니다.
- 실행 순서를 정의합니다.
- 종류:
  - **단순 엣지**: A → B
  - **조건부 엣지**: 상태에 따라 다음 노드를 다르게 선택

### 4) 상태(State)
- 그래프 전체에서 공유되는 데이터입니다.
- 예:
  - 대화 메시지 목록
  - 검색 결과
  - 중간 추론 결과
  - 플래닝 정보
- LangGraph의 중요한 특징은 **“상태를 명시적으로 관리”**한다는 점입니다.

### 5) 상태 전이(State Transition)
- 각 노드가 상태를 읽고, 수정하고, 다음 단계로 넘깁니다.
- 이 방식 덕분에 복잡한 워크플로를 안정적으로 제어할 수 있습니다.

### 6) 조건부 라우팅(Conditional Routing)
- 현재 상태를 보고 다음 실행 노드를 결정합니다.
- 예:
  - 답변이 충분하면 종료
  - 정보가 부족하면 검색 도구 호출
  - 오류가 있으면 재시도

### 7) 사이클/루프
-

In [3]:
# 외부 문서를 직접 제공하면 더 정확한 답변 가능
context = """
LangGraph는 LangChain의 확장으로, 그래프 기반의 멀티 에이전트 워크플로우를 구성할 수 있음
State, Node, Edge 개념을 사용해 복잡한 AI 파이프라인을 구현함
"""

prompt = f"다음 문서를 참고해서 질문에 답하세요.\n\n문서: {context}\n\n질문: {question}"

response = llm.invoke(prompt)
print("[문서 기반 답변]")
print(response.content)

[문서 기반 답변]
LangGraph는 **LangChain의 확장 도구**로, **그래프 기반의 멀티 에이전트 워크플로우**를 구성할 수 있게 해줍니다.

사용하는 주요 개념은 다음과 같습니다:
- **State**: 워크플로우의 현재 상태를 관리
- **Node**: 각 작업이나 처리 단위
- **Edge**: 노드 간의 연결과 흐름을 정의

즉, LangGraph는 이 **State, Node, Edge** 개념을 사용해 복잡한 AI 파이프라인을 구현합니다.


---

## 정리

- RAG는 **외부 문서 → 검색 → 생성** 3단계로 동작함
- 다음 노트북부터 각 단계를 하나씩 실습함

    ```
    02_document_loader   → 문서 불러오기
    03_text_splitter     → 텍스트 분할
    04_embedding         → 임베딩
    05_vector_store      → 벡터 스토어
    06_retriever         → 검색
    07_rag_pipeline      → 전체 파이프라인
    ```